# Lecture 11: RAG Evaluation with RAGAS

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Advanced  

---

## The Big Picture

In Lecture 10 you built a working RAG system. But how do you know  
if it's actually **good**? Is it answering correctly? Is it making things up?

> **"You cannot improve what you do not measure."**  
> — Peter Drucker

Think of RAGAS as a **report card for your RAG system**.  
Just like a school report card grades you in Math, Science, and English,  
RAGAS grades your RAG in Faithfulness, Relevancy, Precision, and Recall.  
Each grade tells you exactly what needs improvement.

### What You Will Learn

| # | Topic | Real-World Analogy |
|---|-------|--------------------|  
| 1 | Why evaluation matters | Why students need exams |
| 2 | Retrieval metrics (precision & recall) | Did you find the right books? |
| 3 | Generation metrics (faithfulness) | Did you copy the answer honestly? |
| 4 | Generation metrics (relevancy & context) | Did you actually answer the question? |
| 5 | What is RAGAS? | The automatic grading machine |
| 6 | Creating test datasets | Writing the exam questions |
| 7 | Hands-on: run a full evaluation | Grade your RAG system! |
| 8 | Common issues & fixes | How to improve each grade |

> **After this lecture:** You'll be able to systematically evaluate  
> your RAG system and know exactly where to improve it.

---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.

**Note:** This lecture requires an **OpenAI API key** (same as Lecture 10).  
RAGAS uses OpenAI under the hood to evaluate your RAG answers.  
The evaluation for 10 questions costs roughly $0.05-0.10.

In [1]:
# Install required packages (run once, then you can skip this cell)
# ragas: the RAG evaluation framework
# datasets: HuggingFace datasets library (used by RAGAS for data format)
%pip install ragas datasets langchain langchain-openai langchain-qdrant langchain-huggingface qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

# Set your OpenAI API key (same as Lecture 10)
# Replace "your-api-key-here" with your actual key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# Verify the key is set
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key and api_key != "your-api-key-here":
    # [:8] shows only the first 8 characters for security
    print(f"API key set: {api_key[:8]}...")
else:
    print("WARNING: Please set your OpenAI API key above!")
    print("RAGAS needs it to evaluate your RAG answers.")

API key set: sk-proj-...


---

## 1. Why Evaluation Matters

RAG systems can fail in **two completely different ways**:

| Failure Mode | What Went Wrong | Example |
|-------------|----------------|---------|  
| **Bad Retrieval** | Retrieved the wrong chunks | Asked about "BERT" but got chunks about "cooking recipes" |
| **Bad Generation** | Right chunks, but LLM hallucinated | Chunks mention 2017, but answer says 2020 |

### Without Evaluation: Guessing Game

```
User: "The answer was wrong!"
You:  "Was it the retrieval or the generation?" 
You:  "...I have no idea." 
```

### With Evaluation: Data-Driven Fixing

```
RAGAS Report:
  Faithfulness:      0.92  (great!)
  Context Recall:    0.45  (problem!)
  
You: "The retrieval is missing relevant chunks.
      I need to increase k or improve chunking."
```

> **Evaluation turns guessing into knowing.**  
> You can pinpoint exactly where your RAG system is weak and fix it.

---

## 2. Retrieval Metrics: Precision & Recall

These metrics measure how well your **retriever** finds the right documents.

### Context Precision

**"Of the chunks I retrieved, how many were actually useful?"**

| Retrieved 5 Chunks | Relevant? | |
|--------------------|-----------|-|
| Chunk about transformers | Yes | |
| Chunk about cooking | No (noise!) | |
| Chunk about BERT | Yes | |
| Chunk about weather | No (noise!) | |
| Chunk about GPT | Yes | |

**Precision = 3 relevant / 5 retrieved = 0.60** (too much noise)

- **High precision (>0.8):** Almost every retrieved chunk is useful
- **Low precision (<0.6):** Too many irrelevant chunks cluttering the context

### Context Recall

**"Of all the relevant chunks that exist, how many did I find?"**

| All Relevant Chunks in DB | Retrieved? | |
|--------------------------|-----------|-|
| Chunk about transformers (Chapter 3) | Yes | |
| Chunk about attention mechanism | No (missed!) | |
| Chunk about BERT (Chapter 3) | Yes | |
| Chunk about self-attention | No (missed!) | |

**Recall = 2 found / 4 total relevant = 0.50** (missing important info)

- **High recall (>0.8):** Found almost everything relevant
- **Low recall (<0.6):** Missing important information

> **Simple way to remember:**  
> **Precision** = low noise (no junk in results)  
> **Recall** = nothing missed (found everything important)

---

## 3. Generation Metric: Faithfulness (The Most Critical!)

**Faithfulness** measures: **Is the answer grounded in the retrieved context?**

This is the **single most important metric** for RAG systems.

| Score | Meaning | Example |
|-------|---------|---------|  
| **1.0** | Every claim in the answer is supported by the context | Perfect! |
| **0.8** | Most claims are supported, minor additions | Good |
| **0.5** | Half the answer is made up | Concerning |
| **0.0** | The answer is completely hallucinated | Terrible! |

### Example

**Context:** "The transformer was introduced by Vaswani et al. in 2017."

| Answer | Faithfulness | Why |
|--------|-------------|------|  
| "The transformer was introduced by Vaswani et al. in 2017." | **1.0** | Directly from context |
| "The transformer was introduced in 2017 and revolutionized NLP." | **0.5** | "revolutionized NLP" not in context |
| "The transformer was created by Google Brain in 2015." | **0.0** | Wrong year, wrong team |

### Target for Production

> **Faithfulness > 0.8 is the minimum for production RAG.**  
> If faithfulness is below 0.8, your RAG system is "lying" to users.  
> Fix this FIRST before worrying about other metrics.

---

## 4. Generation Metrics: Relevancy & Context

### Answer Relevancy

**"Does the answer actually address the question?"**

| Question | Answer | Relevancy |
|----------|--------|-----------|
| "What is BERT?" | "BERT is a language model by Google." | **High** |
| "What is BERT?" | "Transformers use attention." | **Low** (didn't answer the question) |
| "What is BERT?" | "I like pizza." | **0.0** (completely off-topic) |

### All 4 Metrics Summary

| Metric | Measures | Target | What to Fix if Low |
|--------|---------|--------|-------------------|
| **Faithfulness** | Is the answer grounded in context? | > 0.8 | Stronger prompt, temperature=0 |
| **Answer Relevancy** | Does the answer address the question? | > 0.7 | Improve prompt specificity |
| **Context Precision** | Are retrieved chunks useful? | > 0.7 | Add filters, use reranking |
| **Context Recall** | Did we find all relevant chunks? | > 0.7 | Increase k, improve chunking |

> **Think of it like a chain:**  
> Good retrieval (precision + recall) feeds good generation (faithfulness + relevancy).  
> If retrieval is bad, generation can't be good — garbage in, garbage out.

![alt text](image-1.png)

---

## 5. What Is RAGAS?

**RAGAS** = **R**etrieval **A**ugmented **G**eneration **A**ssessment

It's an open-source framework that **automatically evaluates your RAG system**  
using LLMs — no human labelers needed!

### How RAGAS Works

```
You provide:                    RAGAS returns:
                                
  question          ----+
  retrieved context ----+--->   Faithfulness:       0.92
  RAG answer        ----+       Answer Relevancy:   0.88
  ground truth      ----+       Context Precision:  0.75
                                Context Recall:     0.80
```

### What Makes RAGAS Special?

| Feature | Details |
|---------|---------|  
| **Automated** | Uses LLMs to evaluate — no manual scoring |
| **Comprehensive** | Scores both retrieval AND generation |
| **Per-question** | Shows scores for each question individually |
| **Actionable** | Low score on a specific metric tells you exactly what to fix |
| **Industry standard** | Used by top AI teams at major companies |

### The 4 Inputs RAGAS Needs

| Input | What It Is | Where It Comes From |
|-------|-----------|--------------------|
| `question` | The user's question | Your test dataset |
| `contexts` | The retrieved chunks | Your retriever |
| `answer` | The RAG-generated answer | Your RAG chain |
| `ground_truth` | The expected correct answer | You write this manually |

> **Let's build this step by step!**

---

## 6. Step 1: Build the RAG System

Before we can evaluate, we need a RAG system to evaluate!  
Let's rebuild our Lecture 10 RAG chain quickly.

In [ ]:
# ============================================================
# REBUILD THE RAG SYSTEM (same as Lecture 10)
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
# Use the same credentials from previous lectures
# ============================================================

QDRANT_URL = ""
QDRANT_API_KEY = ""

# --- Build Knowledge Base (Lectures 5-8) ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="nlp_course",
)

# --- Build RAG Chain (Lecture 10) ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(f"Knowledge base: {len(chunks)} chunks in Qdrant Cloud")
print(f"RAG chain: retriever | format_docs | prompt | llm | parser")
print(f"\nReady for evaluation!")

c:\Users\mhari\miniconda3\envs\demo1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 384.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base: 21 chunks in Qdrant Cloud
RAG chain: retriever | format_docs | prompt | llm | parser

Ready for evaluation!


In [ ]:
# ============================================================
# REBUILD THE RAG SYSTEM — OpenAI Embeddings Version
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Same RAG system, but with OpenAI embeddings for comparison
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# OPENAI API KEY (use the same key set in cell 3)
# ============================================================
# The key should already be set from cell 3
# If not, uncomment and set it here:
# import os
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# ============================================================

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
QDRANT_URL = "  "
QDRANT_API_KEY = "  "

print("=" * 70)
print("RAG SYSTEM SETUP - OpenAI Embeddings Version")
print("=" * 70)

# --- Build Knowledge Base (Lectures 5-8) ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()
print(f"[SUCCESS] Loaded: {len(documents)} document(s)")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"[SUCCESS] Split into: {len(chunks)} chunks")

# OpenAI embeddings - 1536 dimensions
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"[SUCCESS] OpenAI embeddings initialized (1536 dimensions)")

vectorstore_openai = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings_openai,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="nlp_course_openai",  # Different from free version
)
print(f"[SUCCESS] Vector store created: nlp_course_openai")

# --- Build RAG Chain (Lecture 10) ---
retriever_openai = vectorstore_openai.as_retriever(search_kwargs={"k": 5})


def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain_openai = (
    {"context": retriever_openai | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(f"[SUCCESS] RAG chain assembled (OpenAI embeddings)")
print(f"\n" + "=" * 70)
print("RAG SYSTEM READY - OpenAI Version")
print("=" * 70)
print(f"\nConfiguration:")
print(f"  Embeddings: OpenAI text-embedding-3-small (1536 dims)")
print(f"  Collection: nlp_course_openai")
print(f"  Retriever: k=5")
print(f"  LLM: gpt-4o-mini, temperature=0")
print(f"\n[INFO] You can now evaluate this version with RAGAS")
print(f"[INFO] Compare scores between free and paid embeddings!")

#### What just happened?

We rebuilt the exact same RAG system from Lecture 10 in one cell:
- Loaded and split `nlp_article.txt` into chunks
- Stored chunks in Qdrant with embeddings
- Built the LCEL chain: retriever | format_docs | prompt | llm | parser

Now we have a RAG system to evaluate. Let's write the exam!

---

## 7. Step 2: Create the Test Dataset

To evaluate our RAG, we need **test questions with expected answers**.  
Think of this as writing an exam for your RAG system.

### Two Ways to Create Test Data

| Method | How | Best For |
|--------|-----|----------|
| **Manual** | You write 20-50 questions + expected answers | Most reliable, full control |
| **Synthetic** | RAGAS generates questions from your documents | Fast, covers more ground |

**Best practice:** Start with manual questions (what we'll do today),  
then add synthetic questions later for broader coverage.

### Question Types to Include

| Type | Example | Tests |
|------|---------|-------|
| **Factual** | "When was the transformer introduced?" | Basic retrieval |
| **Descriptive** | "What is Named Entity Recognition?" | Detailed understanding |
| **Multi-concept** | "How do BERT and GPT differ?" | Cross-chunk retrieval |
| **Not in document** | "What is quantum computing?" | Honesty (should say "I don't know") |

In [4]:
# ============================================================
# CREATE THE TEST DATASET — 10 questions + ground truth answers
# ============================================================
# These questions and answers are based on our nlp_article.txt
# The ground_truth is the expected correct answer from the document

test_questions = [
    # Factual questions (simple, single-chunk answers)
    "What is Natural Language Processing?",
    "When was the transformer architecture introduced?",
    "Who developed the GPT models?",

    # Descriptive questions (need more detail)
    "What is Named Entity Recognition?",
    "What is sentiment analysis used for?",

    # Multi-concept questions (may need multiple chunks)
    "What are the main NLP tasks?",
    "How does BERT differ from GPT?",

    # Application questions
    "What is LangChain and what does it do?",
    "What is Retrieval-Augmented Generation?",

    # Best practices
    "What are the best practices for NLP projects?",
]

# Ground truth answers — these come from reading nlp_article.txt
# RAGAS compares the RAG answer against these to measure quality
ground_truths = [
    "NLP is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. It combines computational linguistics with statistical, machine learning, and deep learning models.",

    "The transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. It uses self-attention to process all words simultaneously.",

    "GPT (Generative Pre-trained Transformer) models were developed by OpenAI. GPT-3 has 175 billion parameters and showed emergent abilities like few-shot learning.",

    "Named Entity Recognition (NER) is the task of identifying and classifying named entities in text into categories such as person names, organizations, locations, dates, and monetary values.",

    "Sentiment analysis determines the emotional tone behind text. It is widely used by businesses to understand customer opinions about products and services. It can be performed at document, sentence, or aspect level.",

    "The main NLP tasks include text classification, named entity recognition (NER), sentiment analysis, machine translation, and question answering.",

    "BERT reads text bidirectionally considering both left and right context, while GPT is trained to predict the next word in a sequence giving it text generation capabilities. BERT was released by Google in 2018, GPT was developed by OpenAI.",

    "LangChain is a framework designed to simplify the creation of applications using large language models. It provides a standard interface for chains, integrations with other tools, and end-to-end chains for common applications.",

    "RAG (Retrieval-Augmented Generation) combines retrieval and generation to produce accurate, grounded responses. It first retrieves relevant documents from a knowledge base and then uses those documents as context for the LLM to generate an answer. This reduces hallucination.",

    "Best practices include ensuring data quality, using appropriate evaluation metrics, starting simple and iterating, monitoring model performance in production, and considering ethical implications like bias.",
]

print(f"Test dataset created: {len(test_questions)} questions")
print(f"\nSample:")
# Show first 3 questions and their ground truth answers
# range(3) generates 0, 1, 2
for i in range(3):
    print(f"\n  Q{i + 1}: {test_questions[i]}")
    # [:80] shows first 80 characters of the ground truth
    print(f"  Expected: {ground_truths[i][:80]}...")

Test dataset created: 10 questions

Sample:

  Q1: What is Natural Language Processing?
  Expected: NLP is a branch of artificial intelligence that focuses on the interaction betwe...

  Q2: When was the transformer architecture introduced?
  Expected: The transformer architecture was introduced in the paper 'Attention Is All You N...

  Q3: Who developed the GPT models?
  Expected: GPT (Generative Pre-trained Transformer) models were developed by OpenAI. GPT-3 ...


#### What just happened?

We created a **test dataset** with 10 questions and their expected answers:

- **3 factual questions** — simple facts from the article
- **2 descriptive questions** — need more detailed answers
- **2 multi-concept questions** — may need info from multiple chunks
- **2 application questions** — about LangChain and RAG
- **1 best practices question** — broad topic

The `ground_truths` are the **correct answers** based on reading the article.  
RAGAS will compare the RAG-generated answers against these to calculate scores.

**Tip:** In real projects, aim for 20-50 test questions for reliable evaluation.  
We use 10 here to keep the demo fast and the API costs low.

---

## 8. Step 3: Run RAG on All Test Questions

Now we run our RAG chain on every test question and collect:  
1. The **generated answer** (what the RAG chain produces)  
2. The **retrieved contexts** (what chunks the retriever found)

RAGAS needs both to calculate its metrics.

In [5]:
# ============================================================
# RUN RAG ON ALL TEST QUESTIONS
# ============================================================
# We need to collect: answers + retrieved contexts for each question

answers = []     # Will store the RAG-generated answer for each question
contexts = []    # Will store the retrieved chunks for each question

print("Running RAG chain on all test questions...")
print(f"{'=' * 60}")

# This loop runs the RAG chain on each test question
# enumerate() gives us the index (i=0,1,2...) and the question string
for i, question in enumerate(test_questions):
    # Get the RAG answer
    answer = rag_chain.invoke(question)
    answers.append(answer)

    # Get the retrieved documents (for RAGAS to evaluate retrieval quality)
    # retriever.invoke() returns a list of Document objects
    retrieved_docs = retriever.invoke(question)

    # Extract just the text from each Document object
    # RAGAS expects contexts as a list of strings, not Document objects
    context_texts = [doc.page_content for doc in retrieved_docs]
    contexts.append(context_texts)

    # Show progress (so you know it's working)
    # [:50] shows first 50 characters of the question
    print(f"  [{i + 1}/{len(test_questions)}] {question[:50]}...")

print(f"\nDone! Collected {len(answers)} answers and {len(contexts)} context sets.")

# Preview one result
print(f"\n--- Preview: Question 1 ---")
print(f"Q: {test_questions[0]}")
# [:200] shows first 200 characters of the answer
print(f"A: {answers[0][:200]}...")
print(f"Contexts: {len(contexts[0])} chunks retrieved")

Running RAG chain on all test questions...
  [1/10] What is Natural Language Processing?...
  [2/10] When was the transformer architecture introduced?...
  [3/10] Who developed the GPT models?...
  [4/10] What is Named Entity Recognition?...
  [5/10] What is sentiment analysis used for?...
  [6/10] What are the main NLP tasks?...
  [7/10] How does BERT differ from GPT?...
  [8/10] What is LangChain and what does it do?...
  [9/10] What is Retrieval-Augmented Generation?...
  [10/10] What are the best practices for NLP projects?...

Done! Collected 10 answers and 10 context sets.

--- Preview: Question 1 ---
Q: What is Natural Language Processing?
A: Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective...
Contexts: 5 chunks retrieved


#### What just happened?

We ran our RAG chain on all 10 test questions and collected:

| Collected | What | Shape |
|-----------|------|-------|
| `answers` | RAG-generated answers | List of 10 strings |
| `contexts` | Retrieved chunks per question | List of 10 lists (each has 5 chunk strings) |

Now we have everything RAGAS needs:
- `test_questions` — the questions
- `answers` — what our RAG produced
- `contexts` — what chunks were retrieved
- `ground_truths` — the expected correct answers

Time to grade our RAG system!

---

## 9. Step 4: Run RAGAS Evaluation

Now the exciting part — let's get our RAG system's **report card**!

RAGAS will use an LLM (OpenAI) to automatically evaluate each answer  
across all 4 metrics. This takes about 1-3 minutes for 10 questions.

> **Note:** RAGAS calls OpenAI under the hood to evaluate.  
> This costs a few cents per run. Don't worry — it's very cheap!

In [6]:
# ============================================================
# RUN RAGAS EVALUATION
# ============================================================

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Step 1: Create a HuggingFace Dataset with the required columns
# RAGAS expects a specific format with these exact column names
eval_data = {
    "question": test_questions,       # The test questions
    "answer": answers,                # RAG-generated answers
    "contexts": contexts,             # Retrieved chunks (list of lists)
    "ground_truth": ground_truths,    # Expected correct answers
}

# Dataset.from_dict() creates a HuggingFace Dataset from our dictionary
eval_dataset = Dataset.from_dict(eval_data)

print(f"Evaluation dataset: {len(eval_dataset)} questions")
print(f"Columns: {eval_dataset.column_names}")
print(f"\nRunning RAGAS evaluation (this takes 1-3 minutes)...")
print(f"RAGAS is using OpenAI to score each answer.\n")

# Step 2: Configure the LLM and embeddings for RAGAS
# RAGAS uses these internally for evaluation
ragas_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
ragas_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Step 3: Run the evaluation with explicit LLM and embeddings
result = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness,         # Is the answer grounded in context?
        answer_relevancy,     # Does the answer address the question?
        context_precision,    # Are the retrieved chunks useful?
        context_recall,       # Were all relevant chunks found?
    ],
    llm=ragas_llm,           # Specify LLM for evaluation
    embeddings=ragas_embeddings,  # Specify embeddings for relevancy metric
)

# Step 4: Print the overall scores
print("\n" + "=" * 60)
print("RAG SYSTEM REPORT CARD")
print("=" * 60)

# Convert result to pandas DataFrame to access scores
df_results = result.to_pandas()

# Calculate and display mean scores for each metric
metrics_to_display = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

for metric_name in metrics_to_display:
    if metric_name in df_results.columns:
        # Calculate mean score across all questions
        score = df_results[metric_name].mean()
        
        # Determine if the score is good, okay, or needs work
        if score >= 0.8:
            status = "GREAT"
        elif score >= 0.6:
            status = "OK"
        else:
            status = "NEEDS WORK"

        # :<25 left-aligns the metric name in 25 chars
        # :.4f shows 4 decimal places
        print(f"  {metric_name:<25} {score:.4f}  ({status})")
    else:
        print(f"  {metric_name:<25} N/A")

print(f"\n{'=' * 60}")

C:\Users\mhari\AppData\Local\Temp\ipykernel_1160\625727839.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\mhari\AppData\Local\Temp\ipykernel_1160\625727839.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\mhari\AppData\Local\Temp\ipykernel_1160\625727839.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\mhari\AppData\Local\Temp\ipykernel_1160\625

Evaluation dataset: 10 questions
Columns: ['question', 'answer', 'contexts', 'ground_truth']

Running RAGAS evaluation (this takes 1-3 minutes)...
RAGAS is using OpenAI to score each answer.



Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  25%|██▌       | 10/40 [00:31<01:05,  2.20s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 40/40 [02:10<00:00,  3.26s/it]



RAG SYSTEM REPORT CARD
  faithfulness              0.9000  (GREAT)
  answer_relevancy          0.8307  (GREAT)
  context_precision         1.0000  (GREAT)
  context_recall            0.8600  (GREAT)



#### What just happened?

RAGAS evaluated our RAG system and produced a **report card** with 4 grades:

| Metric | What It Measured | Target |
|--------|-----------------|--------|
| **Faithfulness** | Are the answers grounded in context? (no hallucinations) | > 0.8 |
| **Answer Relevancy** | Do the answers address the questions? | > 0.7 |
| **Context Precision** | Were the retrieved chunks useful? | > 0.7 |
| **Context Recall** | Did we find all relevant information? | > 0.7 |

**How to read your scores:**
- **GREAT (>0.8):** This part of your RAG is working well
- **OK (0.6-0.8):** Room for improvement
- **NEEDS WORK (<0.6):** This is your weakest link — fix this first

Let's dig deeper and see the scores per question.

In [7]:
# ============================================================
# DETAILED PER-QUESTION REPORT
# ============================================================

# Convert RAGAS results to a pandas DataFrame for easy viewing
# .to_pandas() gives us a table with one row per question
df = result.to_pandas()

# Add the original questions to the DataFrame for reference
# The results and questions are in the same order
df['question'] = test_questions
df['answer'] = answers

print("DETAILED SCORES (per question)")
print("=" * 80)

# This loop goes through each row (question) in the results
# iterrows() gives us the index and the row data
for idx, row in df.iterrows():
    # [:50] shows first 50 characters of the question
    print(f"\nQ{idx + 1}: {row['question'][:50]}...")

    # Print each metric score for this question
    # We check if each column exists in case RAGAS version differs
    metrics_to_check = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
    
    for metric in metrics_to_check:
        if metric in df.columns:
            score = row[metric]
            # Handle NaN values that may appear
            if str(score) == 'nan' or score != score:  # score != score checks for NaN
                print(f"    {metric:<25} N/A")
            else:
                print(f"    {metric:<25} {score:.4f}")

print(f"\n{'=' * 80}")
print(f"\nLook for questions with LOW scores — those reveal your RAG's weak spots.")

DETAILED SCORES (per question)

Q1: What is Natural Language Processing?...
    faithfulness              1.0000
    answer_relevancy          0.8540
    context_precision         1.0000
    context_recall            1.0000

Q2: When was the transformer architecture introduced?...
    faithfulness              1.0000
    answer_relevancy          1.0000
    context_precision         1.0000
    context_recall            1.0000

Q3: Who developed the GPT models?...
    faithfulness              1.0000
    answer_relevancy          1.0000
    context_precision         1.0000
    context_recall            1.0000

Q4: What is Named Entity Recognition?...
    faithfulness              1.0000
    answer_relevancy          0.9135
    context_precision         1.0000
    context_recall            1.0000

Q5: What is sentiment analysis used for?...
    faithfulness              1.0000
    answer_relevancy          0.9002
    context_precision         1.0000
    context_recall            1.0000



#### What just happened?

We converted the RAGAS results to a **per-question table** using pandas.

**What to look for:**
- **Questions with low faithfulness** — the LLM is hallucinating for these
- **Questions with low context recall** — the retriever missed relevant chunks
- **Questions with low context precision** — the retriever returned too much noise
- **Questions with low answer relevancy** — the answer didn't address the question

**Each low-scoring question is a clue** about what to fix in your RAG system.  
This is much more useful than just testing a few questions manually!

---

## 10. Common Issues & Target Scores

Here's your **troubleshooting guide** — when a metric is low,  
here's exactly what to fix:

| Problem | Metric | Target | Fix |
|---------|--------|--------|-----|
| RAG is hallucinating | Faithfulness < 0.8 | > 0.8 | Stronger prompt, `temperature=0`, add "only use context" |
| Missing relevant info | Context Recall < 0.7 | > 0.7 | Increase `k` (retrieve more chunks), smaller chunk_size |
| Too much noise in context | Context Precision < 0.6 | > 0.7 | Add metadata filters, use reranking, better embeddings |
| Answer doesn't match question | Answer Relevancy < 0.7 | > 0.7 | More specific prompt, include question in answer template |
| Everything is low | All metrics low | > 0.7 | Check if embeddings match, verify document content |

### Priority Order for Fixing

```
1. Faithfulness    (fix first — your RAG is lying!)
2. Context Recall  (fix second — you're missing information)
3. Context Precision (fix third — too much noise)
4. Answer Relevancy (fix last — usually improves with better context)
```

In [8]:
# ============================================================
# AUTOMATIC DIAGNOSIS — Find the weakest metric
# ============================================================

print("RAG SYSTEM DIAGNOSIS")
print("=" * 60)

# Define the fix recommendations for each metric
# This dictionary maps metric names to their fix suggestions
fixes = {
    "faithfulness": [
        "Add 'Answer ONLY from the context' to your prompt",
        "Set temperature=0 (no creativity for factual RAG)",
        "Use a stronger LLM (e.g., gpt-4o instead of gpt-4o-mini)",
    ],
    "answer_relevancy": [
        "Make your prompt more specific about answering the question",
        "Add 'Be concise and directly answer the question' to prompt",
        "Check if the context contains enough information",
    ],
    "context_precision": [
        "Add metadata filters to narrow retrieval",
        "Use a reranker to push relevant chunks to the top",
        "Try a better embedding model (e.g., OpenAI embeddings)",
    ],
    "context_recall": [
        "Increase k (retrieve more chunks, e.g., k=10)",
        "Use smaller chunk_size (e.g., 300 instead of 500)",
        "Increase chunk_overlap to avoid splitting key info",
    ],
}

# Convert result to DataFrame and calculate mean scores
df_results = result.to_pandas()

# Calculate mean scores for each metric
metric_scores = {}
for metric_name in fixes.keys():
    if metric_name in df_results.columns:
        metric_scores[metric_name] = df_results[metric_name].mean()

if metric_scores:
    # min() with key=metric_scores.get finds the metric with lowest score
    weakest = min(metric_scores, key=metric_scores.get)
    weakest_score = metric_scores[weakest]

    # Show all metrics with status
    for metric_name, score in sorted(metric_scores.items(), key=lambda x: x[1]):
        marker = "  <-- WEAKEST" if metric_name == weakest else ""
        print(f"  {metric_name:<25} {score:.4f}{marker}")

    print(f"\n{'=' * 60}")
    print(f"\nWeakest metric: {weakest} ({weakest_score:.4f})")
    print(f"\nRecommended fixes:")

    # Show the fix suggestions for the weakest metric
    # enumerate() gives us the index and the fix string
    for i, fix in enumerate(fixes[weakest]):
        print(f"  {i + 1}. {fix}")
else:
    print("No metric scores found. Check if RAGAS evaluation completed.")

RAG SYSTEM DIAGNOSIS
  answer_relevancy          0.8307  <-- WEAKEST
  context_recall            0.8600
  faithfulness              0.9000
  context_precision         1.0000


Weakest metric: answer_relevancy (0.8307)

Recommended fixes:
  1. Make your prompt more specific about answering the question
  2. Add 'Be concise and directly answer the question' to prompt
  3. Check if the context contains enough information


#### What just happened?

We built an **automatic diagnosis tool** that:
1. Sorts all metrics from worst to best
2. Identifies the weakest metric
3. Provides specific fix recommendations

This is exactly how you should approach RAG improvement in practice:
1. **Run RAGAS** to get your current scores
2. **Find the weakest metric** (your biggest bottleneck)
3. **Apply the recommended fix**
4. **Run RAGAS again** to measure improvement
5. **Repeat** until all metrics are above target

This is the **data-driven optimization cycle** — much better than guessing!

---

## 11. Creating Test Datasets: Manual vs Synthetic

We created our test questions manually. For larger projects,  
RAGAS can **automatically generate questions from your documents**.

### Synthetic Test Generation (Reference)

```python
# REFERENCE — not runnable without additional setup
from ragas.testset.generator import TestsetGenerator
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Create a test set generator
generator = TestsetGenerator.from_langchain(
    generator_llm=ChatOpenAI(model="gpt-4o-mini"),
    critic_llm=ChatOpenAI(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(),
)

# Generate 20 test questions from your documents
testset = generator.generate_with_langchain_docs(
    documents=chunks,  # Your document chunks
    test_size=20,      # Number of questions to generate
)
```

### Best Practice

| Stage | Approach | Number of Questions |
|-------|---------|--------------------|
| **Start** | Manual questions | 10-20 |
| **Expand** | Add synthetic questions | 50-100 |
| **Validate** | Manually review 10 synthetic questions | Spot check |
| **Production** | Mix of both | 100+ |

> **For this course:** Manual questions are sufficient.  
> Synthetic generation is useful when you have large document collections  
> and need broader test coverage.

---

## 12. Mini Challenges

### Challenge 1: Add More Test Questions
Add 5 more questions to `test_questions` and `ground_truths`.  
Include at least one question whose answer is **NOT** in the document  
(e.g., "What is quantum computing?"). Run RAGAS again and compare scores.

### Challenge 2: Improve Your Weakest Metric
Look at your diagnosis results. Apply the recommended fix  
(e.g., change k, modify the prompt, adjust chunk_size).  
Rebuild the chain and run RAGAS again. Did the score improve?

### Challenge 3: Temperature Comparison
Create two RAG chains — one with `temperature=0` and one with  
`temperature=0.7`. Run RAGAS on both with the same test questions.  
Compare the **faithfulness** scores. Which is more grounded?

> **Hint for Challenge 3:** `temperature=0` should have higher  
> faithfulness because it generates more deterministic, grounded answers.

In [ ]:
# ============================================================
# Challenge 1: Add More Test Questions
# ============================================================
# Add 5 more questions to test_questions and ground_truths
# Include one question NOT in the document
# Run RAGAS again

# Your code here:


In [ ]:
# ============================================================
# Challenge 2: Improve Your Weakest Metric
# ============================================================
# Apply the recommended fix from the diagnosis
# Rebuild the chain and run RAGAS again
# Did the score improve?

# Your code here:


In [ ]:
# ============================================================
# Challenge 3: Temperature Comparison
# ============================================================
# Create two chains: temperature=0 and temperature=0.7
# Run RAGAS on both and compare faithfulness scores

# Your code here:


---

## 13. Quick Reference: RAGAS Evaluation Cheat Sheet

### The 4 RAGAS Metrics

| Metric | Measures | Target | Fix if Low |
|--------|---------|--------|-----------|  
| **Faithfulness** | Is answer grounded in context? | > 0.8 | Stronger prompt, temp=0 |
| **Answer Relevancy** | Does answer address the question? | > 0.7 | More specific prompt |
| **Context Precision** | Are retrieved chunks useful? | > 0.7 | Filters, reranking |
| **Context Recall** | Were all relevant chunks found? | > 0.7 | Increase k, smaller chunks |

### Quick RAGAS Code Pattern

```python
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)
from datasets import Dataset

# 1. Prepare data
eval_data = {
    "question": questions,           # list of strings
    "answer": rag_answers,            # list of strings
    "contexts": retrieved_contexts,   # list of list of strings
    "ground_truth": expected_answers, # list of strings
}

# 2. Create dataset
dataset = Dataset.from_dict(eval_data)

# 3. Evaluate
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy,
             context_precision, context_recall],
)

# 4. View results
print(result)                # Overall scores
df = result.to_pandas()      # Per-question scores
```

### The Optimization Cycle

```
1. Run RAGAS evaluation
2. Find weakest metric
3. Apply targeted fix
4. Run RAGAS again
5. Repeat until all metrics > target
```

---

## 14. Key Takeaways

1. **RAG has 2 failure modes:** bad retrieval OR bad generation — evaluate both separately
2. **RAGAS automates evaluation** — uses LLMs to score, no manual labeling needed
3. **Faithfulness is the #1 metric** — if your RAG is hallucinating, fix this first
4. **Target scores:** faithfulness > 0.8, all others > 0.7
5. **The optimization cycle:** measure, identify weakest metric, fix, re-measure
6. **4 inputs needed:** question, answer, contexts, ground_truth

### The Evaluation Mindset

```
Build RAG (L10) --> Evaluate (L11) --> Identify Issues --> Fix --> Re-evaluate
                         ^                                           |
                         |___________________________________________|     
                              (continuous improvement loop)
```

### Next Lecture

**Lecture 12: Debugging & Fixing Common RAG Problems** — Now that you can  
measure your RAG's performance, let's learn how to fix the most common  
issues and push all metrics above target!

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.  
Here are the rules applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `test_questions`, `ground_truths`, `format_docs()` |
| Constants | `UPPER_CASE` | `OPENAI_API_KEY` (environment variable) |
| Classes | `PascalCase` | `ChatOpenAI`, `Dataset`, `StrOutputParser` (from libraries) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|  
| E401 | One import per line | `from ragas import evaluate` |
| E402 | Imports at the top of each section | All imports at cell top |
| — | Group: stdlib then third-party then local | `os` then `ragas` then `langchain` |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|  
| E225 | Spaces around operators | `score >= 0.8`, `i + 1` |
| E231 | Space after commas | `evaluate(dataset=eval_dataset, metrics=[...])` |
| E265 | Block comments start with `# ` | `# Get the RAG answer` |
| E303 | Two blank lines before function defs | See `format_docs()` definition |
| E501 | Max line length of 79 characters | Long strings use wrapping |

### Best Practices Used

| Practice | Why | Example |
|----------|-----|---------|  
| f-strings | Clean string formatting | `f"Score: {score:.4f}"` |
| `enumerate()` | Index + value in loops | `for i, question in enumerate(test_questions)` |
| List comprehensions | Concise list building | `[doc.page_content for doc in retrieved_docs]` |
| `:<25` formatting | Align columns in output | `f"{metric_name:<25} {score:.4f}"` |
| Docstrings | Explain function purpose | `format_docs()` has a docstring |
| Descriptive names | Code reads like English | `ground_truths` not `gt`, `eval_data` not `ed` |
| `sorted()` with key | Custom sort order | `sorted(scores.items(), key=lambda x: x[1])` |
| Dictionary lookup | Map names to actions | `fixes[weakest]` to get fix recommendations |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
eval_dataset = Dataset.from_dict(eval_data)
result = evaluate(dataset=eval_dataset, metrics=[faithfulness])
for metric_name, score in result.items():
    print(f"{metric_name:<25} {score:.4f}")

# BAD (violates PEP 8)
ed = Dataset.from_dict(eval_data)                  # unclear variable name
r = evaluate(dataset=ed,metrics=[faithfulness])     # no space after comma
for m,s in r.items():                               # single-letter vars
    print(m+": "+str(s))                           # string concat, no f-string
```